# Phase 1

In [2]:
#step 1 initiate a connection with mongo:
from pymongo import MongoClient
import csv
client = MongoClient("mongodb://localhost:27017") #default port
db = client["project_db"] 
collection = db["project_collections"]

In [ ]:
file_path = "variant_summary.txt"
with open(file_path,"r",newline = '',encoding = 'utf-8') as project_file:
    reader = csv.DictReader(project_file,  delimiter='\t') #function used to read a csv document into dictionaries 
    batch = []
    for row in reader:
        batch.append(row)
        if len(batch) == 7500: #took the midpoint of 5000 & 10,000 
            collection.insert_many(batch) #takes the batch of documents and adds it to my collection 
            batch = [] #clears the batch list after adding the previous one for next iteration
    if batch:
        collection.insert_many(batch)
#if the final batch of documents dont reach 7500 it will still be added (so basically this is for the last bit of documents)

In [ ]:
#part1
VariationID_count = list(collection.aggregate([
    {"$match": {"VariationID": {"$exists": True}}},#since mongo is schema-less, i cant guarantee all documents will have the "VariationID" 
    {"$count": "variation_count"}
]))

#the result of the aggregate function is a commandcursor. 
#when converting the result into a dictionary it will result us a list of 1 dictionary
#we want the value of the variation_count of the first (& only) dictionary as a result of my pipeline
print("The total number of variants in the db is: ",VariationID_count[0]["variation_count"])

In [ ]:
#part2
agg_pipe = [{"$group" : {
        "_id" : "$Type",
        "count" : {"$sum" : 1} 
    }},
    ]
result = collection.aggregate(agg_pipe)
for x in result:
    print(x)

In [ ]:
##part3
agg_pipe = [ {
        "$group": {
            "_id": "$Chromosome",
            "count": {"$sum": 1}
        }},
    {
        "$sort": {"count": -1}
    }]

result = collection.aggregate(agg_pipe)
for x in result:
    print(x)

# Phase 2

## reading from mongo to spark

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = r"C:\Users\ASUS\big-data-phase2\.venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\ASUS\big-data-phase2\.venv\Scripts\python.exe"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

print("Python:", sys.executable)
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

Python: C:\Users\ASUS\big-data-phase2\.venv\Scripts\python.exe
HADOOP_HOME: C:\hadoop


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim

spark = SparkSession.builder \
    .appName("ClinVar Phase 2 Cleaning") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .config("spark.mongodb.read.connection.uri", "mongodb://127.0.0.1:27017/project_db.project_collections") \
    .getOrCreate()

df = spark.read\
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "false") \
    .csv(r"C:\Users\ASUS\Desktop\variant_summary.txt.gz")

df.printSchema()
df.show(5)

root
 |-- #AlleleID: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- GeneID: string (nullable = true)
 |-- GeneSymbol: string (nullable = true)
 |-- HGNC_ID: string (nullable = true)
 |-- ClinicalSignificance: string (nullable = true)
 |-- ClinSigSimple: string (nullable = true)
 |-- LastEvaluated: string (nullable = true)
 |-- RS# (dbSNP): string (nullable = true)
 |-- nsv/esv (dbVar): string (nullable = true)
 |-- RCVaccession: string (nullable = true)
 |-- PhenotypeIDS: string (nullable = true)
 |-- PhenotypeList: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginSimple: string (nullable = true)
 |-- Assembly: string (nullable = true)
 |-- ChromosomeAccession: string (nullable = true)
 |-- Chromosome: string (nullable = true)
 |-- Start: string (nullable = true)
 |-- Stop: string (nullable = true)
 |-- ReferenceAllele: string (nullable = true)
 |-- AlternateAllele: string (nullable = true)
 |-- Cytogenetic

## cleaning and transformations

In [2]:
#1 - Cleaning
from pyspark.sql.functions import col, when,trim
# 1. Replace "-" with null
clean_df = df
for column_name in clean_df.columns:
    clean_df = clean_df.withColumn(
        column_name,
        when(trim(col(column_name).cast("string")) == "-", None)
        .otherwise(col(column_name))
    )
    
# 2. Cast data types
clean_df = clean_df \
    .withColumn("#AlleleID", col("#AlleleID").cast("long")) \
    .withColumn("VariationID", col("VariationID").cast("long")) \
    .withColumn("Start", col("Start").cast("long")) \
    .withColumn("Stop", col("Stop").cast("long")) \
    .withColumn("NumberSubmitters", col("NumberSubmitters").cast("int"))

# 3. Filter to one genome assembly
clean_df = clean_df.filter(col("Assembly") == "GRCh38")

# 4. Drop rows with null Chromosome, Start, or Stop
clean_df = clean_df.dropna(subset=["Chromosome", "Start", "Stop"])

# 5. Remove duplicates
clean_df = clean_df.dropDuplicates(["VariationID", "Chromosome", "Start", "Stop"])

# Check final result
clean_df.printSchema()
clean_df.show(10)
print("Rows after cleaning:", clean_df.count())

root
 |-- #AlleleID: long (nullable = true)
 |-- Type: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- GeneID: string (nullable = true)
 |-- GeneSymbol: string (nullable = true)
 |-- HGNC_ID: string (nullable = true)
 |-- ClinicalSignificance: string (nullable = true)
 |-- ClinSigSimple: string (nullable = true)
 |-- LastEvaluated: string (nullable = true)
 |-- RS# (dbSNP): string (nullable = true)
 |-- nsv/esv (dbVar): string (nullable = true)
 |-- RCVaccession: string (nullable = true)
 |-- PhenotypeIDS: string (nullable = true)
 |-- PhenotypeList: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginSimple: string (nullable = true)
 |-- Assembly: string (nullable = true)
 |-- ChromosomeAccession: string (nullable = true)
 |-- Chromosome: string (nullable = true)
 |-- Start: long (nullable = true)
 |-- Stop: long (nullable = true)
 |-- ReferenceAllele: string (nullable = true)
 |-- AlternateAllele: string (nullable = true)
 |-- Cytogenetic: stri

In [3]:
##phase 2 (transformations)
clean_df.select("ClinicalSignificance").distinct() .show(30, truncate=False)

clean_df = clean_df.withColumn("CleanClinicalSignificance",
    when( col("ClinicalSignificance").contains("Pathogenic"),
        "Pathogenic")
    .when(col("ClinicalSignificance").contains("Benign"),
        "Benign")
    .when(col("ClinicalSignificance").contains("Uncertain"),
        "VUS"
    ).otherwise("Other"))
                               
clean_df.select("ClinicalSignificance","CleanClinicalSignificance").show(20, truncate=False)

 ##creating variant_length                              
clean_df = clean_df.withColumn(
    "variant_length",
    col("Stop") - col("Start") + 1)
clean_df.select("Start","Stop", "variant_length").show(10)  
##creating review_score
clean_df.select("ReviewStatus").distinct().show(20, truncate=False)
clean_df = clean_df.withColumn(
    "review_score",when(
        col("ReviewStatus").contains("practice guideline"),4)
    .when(col("ReviewStatus").contains("reviewed by expert panel"),3)
    .when( col("ReviewStatus").contains("multiple submitters"),2)
    .when(col("ReviewStatus").contains("single submitter"),1)
    .otherwise(0))
clean_df.select("ReviewStatus","review_score").show(20, truncate=False)
##removing vus and ambiguous rows 
transformed_df = clean_df.filter((col("CleanClinicalSignificance") == "Pathogenic") |
    (col("CleanClinicalSignificance") == "Benign"))
transformed_df = transformed_df.withColumn("is_pathogenic",
    when(col("CleanClinicalSignificance") == "Pathogenic", 1)
    .otherwise(0))
transformed_df.select("CleanClinicalSignificance","is_pathogenic").show(20)

+----------------------------------------------------------------+
|ClinicalSignificance                                            |
+----------------------------------------------------------------+
|Likely risk allele                                              |
|Benign/Likely benign                                            |
|Likely pathogenic/Likely pathogenic, low penetrance             |
|Pathogenic/Likely risk allele                                   |
|Conflicting classifications of pathogenicity; other             |
|Conflicting classifications of pathogenicity; other; risk factor|
|Likely pathogenic, low penetrance                               |
|confers sensitivity                                             |
|Likely benign                                                   |
|Affects                                                         |
|Uncertain significance/Uncertain risk allele                    |
|Uncertain significance                                       

In [ ]:
from pyspark.sql.functions import col, count, sum, when, round

#creating a df it will include sum of total gene varients, the pathogenic count, and the conflict count for each gene symbol.
#count(*) will count the number of rows with the same gene symbol.
#if the clinical significance is patho, add 1 to the sum, else just add 0, using alias we will create a new column.

genes_df = transformed_df.groupBy("GeneSymbol").agg(
    count("*").alias("gene_total_variants"),
    sum(when(col("CleanClinicalSignificance") == "Pathogenic", 1).otherwise(0)).alias("gene_pathogenic_count"),
    sum(when(col("ClinicalSignificance").contains("Conflicting"), 1).otherwise(0)).alias("gene_conflict_count")
)


#i created a sumbmissions df, with a variation id column bc i want to join two dfs on it later on.
#the id column must have unique values only, so any duplicates must be removed thus used drop duplicted. 
submissions_df = transformed_df.select(
    "VariationID"
).dropDuplicates(["VariationID"])


#as required i joined the df's, i chose left in-specific bc it will ensure that if the transformed df[GeneSymbol] column has a value of unknown, i dont wantit to be ingnored
#i want it to become just null when joining the two df's, this is the safest option to not lose any data from all the different methods of joining
joined_df = transformed_df.join(
    genes_df,
    on="GeneSymbol",
    how="left"
)

#same explaination for this part
final_df = joined_df.join(
    submissions_df,
    on="VariationID",
    how="left"
)

submissions_df.show(2)
joined_df.show(2)
final_df.show(2)

## analytical questions

**question 1**

In [7]:
#using the count function most pathogenic variants.
#create a df where i have the genesymbol, and the count for it.
#must sort (using orderBy) it from largest (descending) and my limit 10.

top_10_genes = final_df.filter(
    col("CleanClinicalSignificance") == "Pathogenic"
).groupBy("GeneSymbol").agg(
    count("*").alias("varient_count")
).orderBy(
    col("varient_count").desc()
).limit(10)

top_10_genes.show(10)

+----------+-------------+
|GeneSymbol|varient_count|
+----------+-------------+
|     BRCA2|         5276|
|       NF1|         4865|
|     BRCA1|         3971|
|       ATM|         3160|
|      FBN1|         2477|
|       APC|         2335|
|      MSH6|         2035|
|       DMD|         1938|
|      MSH2|         1896|
|      PKD1|         1634|
+----------+-------------+



**question 2**

In [8]:
#im going to need to know how many of the types are pathogenic and bengin.
#found the count for each type and significance and added it to a new column.

varients_disitrbusion = final_df.groupBy(
    "Type",
    "CleanClinicalSignificance"
).agg(
    count("*").alias("variant_count")
)

varients_disitrbusion.show()

+--------------------+-------------------------+-------------+
|                Type|CleanClinicalSignificance|variant_count|
+--------------------+-------------------------+-------------+
|           Inversion|                   Benign|           55|
|         Duplication|                   Benign|         9972|
|single nucleotide...|               Pathogenic|       110817|
|            Deletion|               Pathogenic|        68910|
|      Microsatellite|                   Benign|         6130|
|           Inversion|               Pathogenic|           78|
|    copy number gain|               Pathogenic|         1462|
|             Complex|               Pathogenic|            4|
|    copy number loss|                   Benign|         1156|
|         Duplication|               Pathogenic|        27891|
|            Deletion|                   Benign|        12971|
|               Indel|                   Benign|          329|
|       Translocation|               Pathogenic|       

**question 3**

In [9]:
#group the genesym + find count using sum
#filter them out by check the ones with a positive conflict
#pathogenic ratio formula: (pathogenic count/ total variants count), rounding it to 2

gene_stats = clean_df.groupBy("GeneSymbol").agg(
    count("*").alias("gene_total_variants"),
    sum(
        when(col("CleanClinicalSignificance") == "Pathogenic", 1).otherwise(0)
    ).alias("gene_pathogenic_count"),
    sum(
        when(col("ClinicalSignificance").contains("Conflicting"), 1).otherwise(0)
    ).alias("gene_conflict_count")
)

positive_conflicts = gene_stats.filter(
    col("gene_conflict_count") > 0
).withColumn(
    "pathogenic_ratio",
    round(col("gene_pathogenic_count") / col("gene_total_variants"), 2)
)

positive_conflicts.show()

+----------+-------------------+---------------------+-------------------+----------------+
|GeneSymbol|gene_total_variants|gene_pathogenic_count|gene_conflict_count|pathogenic_ratio|
+----------+-------------------+---------------------+-------------------+----------------+
|       NF1|              16903|                 4865|               1261|            0.29|
|   RNASET2|                196|                    9|                  9|            0.05|
|     ABCC6|               1817|                  175|                 80|             0.1|
|      MSR1|                136|                    2|                  2|            0.01|
|     KCNH5|                851|                    6|                 21|            0.01|
|      AFF2|                559|                    7|                 10|            0.01|
|     DARS2|                507|                   46|                 35|            0.09|
|     AARS1|               1679|                   60|                 80|      

**question 4**

In [10]:
highest_proportions = transformed_df.groupBy("PhenotypeList").agg(
    count("*").alias("total_variants"),
    sum(when(col("CleanClinicalSignificance") == "Pathogenic", 1).otherwise(0)).alias("pathogenic_variants")
).withColumn(
    "pathogenic_proportion",
    round(col("pathogenic_variants") / col("total_variants"), 4)
).orderBy(col("pathogenic_proportion").desc()).limit(10)

highest_proportions.show()

+--------------------+--------------+-------------------+---------------------+
|       PhenotypeList|total_variants|pathogenic_variants|pathogenic_proportion|
+--------------------+--------------+-------------------+---------------------+
|Inborn genetic di...|             1|                  1|                  1.0|
|Global developmen...|             1|                  1|                  1.0|
|Encephalopathy du...|            26|                 26|                  1.0|
|Niemann-Pick dise...|             1|                  1|                  1.0|
|Peroxisome biogen...|             3|                  3|                  1.0|
|not provided|WFS1...|             1|                  1|                  1.0|
|Autosomal recessi...|             4|                  4|                  1.0|
|Oculocutaneous al...|             1|                  1|                  1.0|
|not provided|GM1 ...|             1|                  1|                  1.0|
|not provided|Fami...|             1|   

# Phase 3

## preparing DF for model

In [8]:
from pyspark.sql.functions import col, when, count, sum as spark_sum, abs as spark_abs
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

#first thing, create a separte df where it has 0,1 if its patho or benign
#the model is to be chosen later on, but its binary so most likely logisticregression or maybe random forest
#try to find the most "relavent" columns/features that i think will help the model to get accurate results in finding if its patho or not
#must find the correlation between the different feature and the target to see if they are relavent to keep them as a feature or if they can be dropped.


patho_beni_df = transformed_df.filter(
    (col("CleanClinicalSignificance") == "Pathogenic") |
    (col("CleanClinicalSignificance") == "Benign")
)

#now adding a new column into the new df to see if the row is patho or not, where it will be the target column.
patho_beni_df = patho_beni_df.withColumn(
    "is_pathogenic",
    when(col("CleanClinicalSignificance") == "Pathogenic", 1).otherwise(0)
)

#selecting the potentially important columns.
#this step is very important, we can ask a domain professional to tell us which features are important and arent so we have an accurate model.

model_df = patho_beni_df.select(
    "is_pathogenic",
    "variant_length",
    "review_score",
    "NumberSubmitters",
    "Type",
    "Chromosome"
)

#this step i can take 2 approaches, look at the number of rows in the df that have na and drop them IF AND ONLY IF they arent alot
#second approach which i decided to continue on is to fill the missing values with 0 and unknown, bc its faster. 

model_df = model_df.fillna({
    "variant_length": 0,
    "review_score": 0,
    "NumberSubmitters": 0
})

model_df = model_df.fillna({
    "Type": "Unknown",
    "Chromosome": "Unknown"
})

#after i chose the columns that might be needed, ill find the correlation between them and the target column to see if they really are important or not.
#since i decided i wont be dropping the rows that contain na values, so the code dosnt crash i will handel the invalid by adding them to another category
#string indexer will convert all categorical values to numerical, find the most frequent and based on that it will label it by a numerical value.
#best practice of using string indexer is with hot label encoding after it, but i wont do that
#i applied the string indexer on only 2 of the columns since they are the onlly caetgorical ones

type_corr_indexer = StringIndexer(
    inputCol="Type",
    outputCol="Type_corr_index",
    handleInvalid="keep"
)

type_corr_model = type_corr_indexer.fit(model_df)
correlation_df = type_corr_model.transform(model_df)


chromosome_corr_indexer = StringIndexer(
    inputCol="Chromosome",
    outputCol="Chromosome_corr_index",
    handleInvalid="keep"
)

chromosome_corr_model = chromosome_corr_indexer.fit(correlation_df)
correlation_df = chromosome_corr_model.transform(correlation_df)

columns_to_check = [
    "variant_length",
    "review_score",
    "NumberSubmitters",
    "Type_corr_index",
    "Chromosome_corr_index"
]

#now i can be sure if they columns actually matter in the model or not by printing the result:
print("Correlation of each column with is_pathogenic:")
for column_name in columns_to_check:
    corr_value = correlation_df.stat.corr(column_name, "is_pathogenic")
    print(column_name, ":", corr_value)



#now since i have to transform and fit many things i decided to make things easier by building a pipeline
#i will convert any categorical columns to numeric using string indexer
#i will then add all the features to a single vector since the ML model only works on 2 columns, this is odnt by vector assembler
#then finally build the pipeline, where i will add all stages then finally fit and finally tranform.

type_indexer = StringIndexer(
    inputCol="Type",
    outputCol="Type_index",
    handleInvalid="keep"
)

chromosome_indexer = StringIndexer(
    inputCol="Chromosome",
    outputCol="Chromosome_index",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=[
        "variant_length",
        "review_score",
        "NumberSubmitters",
        "Type_index",
        "Chromosome_index"
    ],
    outputCol="features"
)

prep_pipeline = Pipeline(stages=[
    type_indexer,
    chromosome_indexer,
    assembler
])

prep_model = prep_pipeline.fit(model_df)

prepared_ml_df = prep_model.transform(model_df)
prepared_ml_df = prepared_ml_df.withColumnRenamed("is_pathogenic", "label")
prepared_ml_df = prepared_ml_df.select("features", "label")


#final check to just make sure the df only has 2 columns features and the target/label before sending it off to spliting into testing and training
prepared_ml_df.show(5, truncate=False)
prepared_ml_df.printSchema()

Correlation of each column with is_pathogenic:
variant_length : 0.0455810201003526
review_score : -0.2065647111200072
NumberSubmitters : -0.07559504166807947
Type_corr_index : 0.27390723296276354
Chromosome_corr_index : -0.05688257465467963
+-----------------------+-----+
|features               |label|
+-----------------------+-----+
|[2.0,1.0,2.0,5.0,20.0] |1    |
|[1.0,2.0,10.0,0.0,14.0]|1    |
|[1.0,1.0,3.0,0.0,0.0]  |1    |
|[4.0,0.0,1.0,1.0,19.0] |1    |
|[1.0,0.0,1.0,0.0,1.0]  |1    |
+-----------------------+-----+
only showing top 5 rows
root
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = false)



## building the model

In [9]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Now we are splitting the data into training and testing sets, and creating random forest classifier. 
#the best ratio for the training and testing df's is 80% , 20% only whne the data set is large, which we have.

training_df, testing_df = prepared_ml_df.randomSplit([0.8, 0.2],seed=42)
print("Training rows:", training_df.count())
print("Testing rows:", testing_df.count())

#now ill be giving the model the training data set, then ill work on the ParamGridBuilder to tune and optimize the paramteres.
 
random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",)

#maxdepth might cause overfitting if i intrduce to it many deep trees thus im trying to be fair and give it abit more
#number of trees will try all the different combinations of the depth, so 20 trees with 3 small, 20 with 6 medium, and 20 with 8
#its best not to have it many number of trees since it will cause the training duration to take long. 

param_grid = ParamGridBuilder() \
    .addGrid(random_forest.numTrees, [10,30]) \
    .addGrid(random_forest.maxDepth, [3,6,8]) \
    .build()


#i decided to combine the binary class. evaluator when creating the model since it will give me thebest model during tunning
#i would've used multiclassclassfication, but i will use it to evalute my model, but now im trying to bulid the best model possible

binary_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC")


cross_validator = CrossValidator(
    estimator=random_forest, #model type
    estimatorParamMaps=param_grid, #settings we are trying our model on
    evaluator=binary_evaluator, #evaluating the model to choose the best one using AUC-ROC
    numFolds=3, #testing the "setting" 3 times on different training splits
    parallelism=2, #how many settings can be done by the random_forest model at a single time
    seed=42) #trying to avoid randomness as much as possible by setting the seed)

#now we can acutally build the model on the training data we have
final_model = cross_validator.fit(training_df)

#giving the model or testing data.
predictions = final_model.transform(testing_df)

#random_forest tree will automatically add a new column called the probability but i wont add it. (useless)
predictions.select("label", "prediction").show(10, truncate=False)

Training rows: 404713
Testing rows: 101342
+-----+----------+
|label|prediction|
+-----+----------+
|0    |1.0       |
|0    |1.0       |
|0    |1.0       |
|0    |1.0       |
|0    |1.0       |
|0    |1.0       |
|1    |1.0       |
|1    |1.0       |
|1    |1.0       |
|1    |1.0       |
+-----+----------+
only showing top 10 rows


## evaluating the model

In [23]:
#i already created the binary classification evaluator (AUC-ROC)
#it will evaluate how well the model can distingush betweent the two classes (patho,beign)
#the result will be from 1-0, 1 being a perfect well trained model, and anything below .6 is considered weak.
binary_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = binary_evaluator.evaluate(predictions)

#now using multi class with different metrics
#1. being the accuracy. it will tell us how accurate our model is by compating the actual value and the predicted value
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

#2. the precision metric: gives us how many predictions from each class where correct (not a whole number)
#uses a formula: true positive / true positive + false positive
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)


#3. recall metric evaluation: measures how many of the actual members of a specific class your model successfully managed to find.
#uses a different formula for the precision: true positive / true positive + false negative
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)


#4. f1 metric: its a metric built on the recall and precision, so to be able to use it, evaluate using recall and precision first.
#uses a formula: (2* (precision * recall) / (precision + recall))
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

#now actually using these metrices to evalute my model
#then printing the results ot see if my model is good enough or not
accuracy_multiclass = accuracy_evaluator.evaluate(predictions)
precision_multiclass = precision_evaluator.evaluate(predictions)
recall_multiclass = recall_evaluator.evaluate(predictions)
f1_multiclass = f1_evaluator.evaluate(predictions)

print("the binary matric evaluation of the model:", auc)
print("the accuracy matric evaluation of the model:", accuracy_multiclass)
print("precision matric evaluation of the model:", precision_multiclass)
print("the recall matric evaluation of the model:", recall_multiclass)
print("the f1 Score:", f1_multiclass)


#confusion matrix:
print("Confusion Matrix:")
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

#now getting the best random forest model that was chosen by the cross validator
best_rf_model = final_model.bestModel

#checking feature importance to see which columns affected the model predictions the most
feature_names = [
    "variant_length",
    "review_score",
    "NumberSubmitters",
    "Type_index",
    "Chromosome_index"
]

#this gets the importance score for each feature from the trained random forest model
#then priting the importance score for each column/feature
importances = best_rf_model.featureImportances

print("Feature Importance:")
for i in range(len(feature_names)):
    name = feature_names[i]
    score = importances[i]
    print(name, ":", float(score))

#now showing a few sample predictions to compare the actual label with the predicted label
#probability shows how confident the model was for each class (comes be deault from random forest model)
predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

the binary matric evaluation of the model: 0.7844034236619046
the accuracy matric evaluation of the model: 0.7356180063547196
precision matric evaluation of the model: 0.7425173328655787
the recall matric evaluation of the model: 0.7356180063547196
the f1 Score: 0.7297654631434907
Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|47162|
|    0|       1.0| 7867|
|    1|       0.0|18926|
|    1|       1.0|27387|
+-----+----------+-----+

Feature Importance:
variant_length : 0.0776511057754963
review_score : 0.19206516317522748
NumberSubmitters : 0.043093216705749955
Type_index : 0.6649353892427676
Chromosome_index : 0.022255125100758555
+-----+----------+---------------------------------------+
|label|prediction|probability                            |
+-----+----------+---------------------------------------+
|0    |1.0       |[0.3361615004174173,0.6638384995825827]|
|0    |1.0       |[0.3361615004174173,0.6638384995825827]|
|